In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../air_quality_complete_dataset.parquet")

In [3]:
print(df.shape)
print(df.columns)
df.head()

(50417, 85)
Index(['Samplingpoint', 'Pollutant', 'Start', 'End', 'Value', 'Unit',
       'AggType', 'Validity', 'Verification', 'ResultTime', 'DataCapture',
       'FkObservationLog', 'station_id', 'pollutant_code', 'source_file',
       'Country', 'B-G Namespace', 'Year', 'Air Quality Network',
       'Air Quality Network Name', 'Timezone', 'Air Quality Station EoI Code',
       'Air Quality Station Nat Code', 'Air Quality Station Name',
       'Sampling Point Id', 'Air Pollutant', 'Longitude', 'Latitude',
       'Altitude', 'Altitude Unit', 'Air Quality Station Area',
       'Air Quality Station Type', 'Operational Activity Begin',
       'Operational Activity End', 'Sample Id', 'Inlet Height',
       'Inlet Height Unit', 'Building Distance', 'Building Distance Unit',
       'Kerb Distance', 'Kerb Distance Unit', 'Distance Source',
       'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions',
       'Heating Emissions Unit', 'Mobile', 'Traffic Emissions',
       'Traff

,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,...,Detection Limit,Detection Limit Unit,Documentation,QA Report,Duration,Duration Unit,Cadence,Cadence Unit,Source Data URL,Imported
0,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-01-30,2025-01-31,29.000000000000000000,ug.m-3,day,1,2,2025-01-31 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
1,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-01-31,2025-02-01,27.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
2,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-01,2025-02-02,5.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
3,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-02,2025-02-03,10.000000000000000000,ug.m-3,day,1,2,2025-02-03 09:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21
4,IT/SPO.IT0470A_6001_BETA_2013-12-20_00:00:00,6001,2025-02-03,2025-02-04,26.000000000000000000,ug.m-3,day,1,3,2025-02-04 08:00:00,...,1.0,ug.m-3,Evaluation of uncertainty in progress,in preparation,1440,minute,1440,minute,http://cdr.eionet.europa.eu/it/eu/aqd/d/envanp...,2025-10-01 14:12:21


In [4]:
cols_to_keep = [
    "station_id",
    "Pollutant",
    "Start",
    "Value",
    "Unit",
    "Country",
    "Municipality",
    "Longitude",
    "Latitude",
    "Altitude",
    "Validity",
    "Verification"
]

df = df[cols_to_keep]

In [ ]:
# Clean dataset
df = df.dropna(subset=["Value"])
df['y'] = pd.to_numeric(df['y'], errors='coerce')
df = df[df['y'] > 0]
df = df.drop(columns=['Validity', 'Verification'])
df["Start"] = pd.to_datetime(df["Start"])
df = df.rename(columns={"Value": "y"})
df = df[(df["Validity"] == 1) & (df["Verification"] >= 2)]
df = df.copy()

In [ ]:
# drop stations with too few observations
counts = df.groupby("station_id").size()
valid_stations = counts[counts >= 30].index
df = df[df["station_id"].isin(valid_stations)]
df.groupby("station_id").size().describe()

In [ ]:
# time variables
df["Start"] = pd.to_datetime(df["Start"])
df = df.rename(columns={"Value": "y"})
df["station_idx"] = df["station_id"].astype("category").cat.codes
df["month"] = df["Start"].dt.month
df["day_of_week"] = df["Start"].dt.dayofweek

In [17]:
df.to_csv("pm25_bayes.csv", index=False)